# drfn-plain-seeds: the no-label baseline at two more seeds

"Labels help translation" (+1.58 chrF with beam decoding) is our largest
translation result, but its no-label baseline `plain_bpe` had one seed (11).
This trains seeds 22 and 33 in parallel (one per GPU), same cleaned data,
tokenizer and settings, then decodes with beam 5 + no-repeat 4-gram.

In [ ]:
!pip install -q tokenizers sacrebleu

In [ ]:
import glob, os, shutil, subprocess
def find(pat):
    hits = sorted(glob.glob('/kaggle/input/**/' + pat, recursive=True))
    if not hits: raise RuntimeError('not found under /kaggle/input: ' + pat)
    return hits[0]
PACK = '/kaggle/working/pack'
if not os.path.exists(PACK): shutil.copytree(os.path.dirname(find('run_clean.sh')), PACK)
ev = os.path.dirname(find('run_eval2.sh'))
for f in ('beam.py', 'slot_tests.py', 'eval2.py'):
    shutil.copy(os.path.join(ev, f), PACK)
os.chdir(PACK); print(os.listdir(PACK))

In [ ]:
import torch
n = torch.cuda.device_count(); print('GPUs', n)
procs = []
for k, seed in enumerate((22, 33)):
    env = dict(os.environ, SEED=str(seed), LABELS='geezrpa', VARIANTS='plain_bpe',
               CUDA_VISIBLE_DEVICES=str(k % max(n, 1)))
    log = open(f'/kaggle/working/train_s{seed}.txt', 'w')
    procs.append(subprocess.Popen(['bash', 'run_clean.sh'], env=env, stdout=log, stderr=subprocess.STDOUT))
    if n < 2: procs[-1].wait()
print('exit codes', [p.wait() for p in procs])
for seed in (22, 33):
    print(seed, [l for l in open(f'/kaggle/working/train_s{seed}.txt') if 'final eval' in l or 'Error' in l][-3:])

In [ ]:
print('exit', subprocess.call(['python', '-u', 'eval2.py', '--ckpt-root', 'runs',
                               '--out', '/kaggle/working/eval2']))
import json
for p in sorted(glob.glob('/kaggle/working/eval2/*/eval2.json')):
    r = json.load(open(p)); d = r['decode']
    print(r['run'], 'beam5_nr4 chrF %.2f  BLEU %.2f  (best epoch %s)' % (
        d['beam5_nr4']['chrF_char'], d['beam5_nr4']['BLEU'], r['best_epoch']))